# Trying to create the chunker for the DOCX files, for libre-office writer, ms word, google docs and so on 

In [24]:
import pypandoc
from pathlib import Path
import os
import subprocess

In [34]:
nolen_path = Path(
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.pptx"
)

In [12]:
pypandoc.download_pandoc()

In [35]:
nolen_path = Path(nolen_path)
ext = nolen_path.suffix.lower()

temp_docx = None

if ext == ".doc":
    subprocess.run(
        [
            "soffice",
            "--headless",
            "--convert-to",
            "docx",
            "--outdir",
            str(nolen_path.parent),
            str(nolen_path),
        ],
        check=True,
    )

    temp_docx = nolen_path.with_suffix(".docx")

    if not temp_docx.exists():
        raise FileNotFoundError(
            f"LibreOffice conversion failed: {temp_docx}"
        )

    target_input = str(temp_docx)
else:
    target_input = str(nolen_path)

markdown_test = pypandoc.convert_file(target_input, to="markdown")

print(markdown_test)

if temp_docx and os.path.exists(temp_docx):
    os.remove(temp_docx)

## A PPTX result {#slide-1}

This is the PPTX content

## The Second page {#slide-2}

![](ppt/media/image1.png)



In [43]:
import zipfile
import os

def extract_office_images(file_path: str, output_dir: str = "extracted_images") -> list:
    """
    Extracts all images directly from a .docx or .pptx file.
    Works for:
      - Word files (stored in word/media/)
      - PowerPoint files (stored in ppt/media/)
    """
    extracted_files = []
    
    with zipfile.ZipFile(file_path, 'r') as z:
        for member in z.namelist():
            # Check for images inside Word or PowerPoint internal media folders
            if member.startswith(('word/media/', 'ppt/media/')) and not member.endswith('/'):
                # Extract image file to target output directory
                z.extract(member, path=output_dir)
                full_path = os.path.join(output_dir, member)
                extracted_files.append(full_path)
                
    return extracted_files

In [54]:
# ---------------------------------------------------------
# Mocking your existing functions (Replace with actual implementations)
# ---------------------------------------------------------
def Save_Image(image_path: str):
    """Saves or moves the extracted image to your designated storage."""
    print(f"[Log] Image saved permanently: {image_path}")


def Vision_LLM(image_path: str) -> str:
    """Calls your Vision Model and returns the image description."""
    print(f"[Log] Calling Vision LLM for: {image_path}")
    return "This is an AI-generated description of the image."



In [67]:
import os
import time
import random
import subprocess
import urllib.parse
import re
import zipfile
import pypandoc
from pptx import Presentation



# ---------------------------------------------------------


class UniversalDocumentParser:
    def __init__(self, media_temp_dir: str = "temp_media"):
        self.media_temp_dir = media_temp_dir
        pypandoc.download_pandoc()

        self.legacy_extensions = {
            ".doc": "docx",
            ".ppt": "pptx",
            ".odp": "pptx",
            ".odt": "docx",  
        }

    def _generate_unique_filename(self, original_path: str, new_ext: str) -> str:
        """Generates {original_name}_{time_in_ms}_{random}.{new_ext}"""
        base_name = os.path.splitext(os.path.basename(original_path))[0]
        time_ms = int(time.time() * 1000)
        random_num = random.randint(1000, 9999)
        return f"{base_name}_{time_ms}_{random_num}.{new_ext}"

    def _convert_legacy_file(self, filepath: str, ext: str) -> str:
        """Converts legacy files using LibreOffice headless mode."""
        target_ext = self.legacy_extensions[ext]
        file_dir = os.path.dirname(filepath) or "."

        print(f"[System] Converting legacy {ext} to {target_ext}...")
        try:
            subprocess.run(
                [
                    "soffice",
                    "--headless",
                    "--convert-to",
                    target_ext,
                    "--outdir",
                    file_dir,
                    filepath,
                ],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                timeout=60,
            )
        except subprocess.TimeoutExpired:
            raise RuntimeError(f"Conversion of {filepath} timed out.")
        except FileNotFoundError:
            raise EnvironmentError("LibreOffice 'soffice' command not found.")

        base_name = os.path.splitext(os.path.basename(filepath))[0]
        default_converted_path = os.path.join(file_dir, f"{base_name}.{target_ext}")

        unique_filename = self._generate_unique_filename(filepath, target_ext)
        unique_filepath = os.path.join(file_dir, unique_filename)

        if os.path.exists(default_converted_path):
            os.rename(default_converted_path, unique_filepath)
            print(f"[System] Converted file saved uniquely as: {unique_filepath}")
            return unique_filepath

        return filepath

    def extract_office_images(self, zip_filepath: str, output_dir: str) -> dict:
        """
        Extracts images directly from .docx / .pptx zip structure.
        Returns a mapping of { image_filename: full_extracted_path }.
        """
        extracted_map = {}
        if not zipfile.is_zipfile(zip_filepath):
            return extracted_map

        with zipfile.ZipFile(zip_filepath, "r") as z:
            for member in z.namelist():
                if member.startswith(
                    ("word/media/", "ppt/media/", "Pictures/")
                ) and not member.endswith("/"):
                    z.extract(member, path=output_dir)
                    full_path = os.path.join(output_dir, member)
                    filename = os.path.basename(member)
                    extracted_map[filename] = full_path

        return extracted_map

    def _extract_pptx_charts_as_markdown(self, pptx_path: str) -> list:
        """Reads PowerPoint charts via python-pptx and formats their numerical data as Markdown tables."""
        chart_tables = []
        try:
            prs = Presentation(pptx_path)
            for slide_num, slide in enumerate(prs.slides, start=1):
                for shape in slide.shapes:
                    if shape.has_chart:
                        chart = shape.chart
                        title = (
                            chart.chart_title.text_frame.text
                            if chart.has_title
                            else f"Slide {slide_num} Chart"
                        )

                        categories = [str(c.label) for c in chart.plots[0].categories]
                        series_list = chart.series

                        # Construct Markdown Table header
                        series_names = [
                            s.name if s.name else f"Series {i+1}"
                            for i, s in enumerate(series_list)
                        ]
                        md_table = (
                            f"\n\n### [Chart Data: {title} (Slide {slide_num})]\n"
                        )
                        md_table += "| Category | " + " | ".join(series_names) + " |\n"
                        md_table += (
                            "|---| " + " | ".join(["---"] * len(series_names)) + " |\n"
                        )

                        # Construct rows
                        for cat_idx, cat in enumerate(categories):
                            row_vals = []
                            for series in series_list:
                                try:
                                    row_vals.append(str(series.values[cat_idx]))
                                except IndexError:
                                    row_vals.append("-")
                            md_table += f"| {cat} | " + " | ".join(row_vals) + " |\n"

                        chart_tables.append(md_table)
        except Exception as e:
            print(f"[Warning] Could not extract PPTX chart data: {e}")

        return chart_tables

    def _extract_docx_charts_as_markdown(self, docx_path: str) -> list:
        """Reads Word charts directly from the DOCX XML zip structure."""
        chart_tables = []
        if not zipfile.is_zipfile(docx_path):
            return chart_tables

        try:
            import xml.etree.ElementTree as ET
            with zipfile.ZipFile(docx_path, "r") as z:
                # Find all chart XML files in the DOCX archive
                chart_files = [f for f in z.namelist() if f.startswith('word/charts/chart') and f.endswith('.xml')]

                for chart_num, chart_file in enumerate(chart_files, start=1):
                    xml_content = z.read(chart_file)
                    root = ET.fromstring(xml_content)

                    # OpenXML drawingml namespace
                    ns = {'c': 'http://schemas.openxmlformats.org/drawingml/2006/chart'}

                    # 1. Get Chart Title
                    title_node = root.find('.//c:title//c:tx//c:v', ns)
                    title = title_node.text if title_node is not None else f"Chart {chart_num}"

                    # 2. Extract Series (columns)
                    series_nodes = root.findall('.//c:ser', ns)
                    if not series_nodes:
                        continue

                    # 3. Extract Categories (rows) from the first series
                    categories = []
                    cat_nodes = series_nodes[0].findall('.//c:cat//c:pt/c:v', ns)
                    for cat in cat_nodes:
                        categories.append(cat.text or "")

                    series_names = []
                    series_data = []

                    for ser in series_nodes:
                        # Get series name
                        tx_node = ser.find('.//c:tx//c:v', ns)
                        s_name = tx_node.text if tx_node is not None else f"Series {len(series_names)+1}"
                        series_names.append(s_name)

                        # Get series values
                        val_nodes = ser.findall('.//c:val//c:pt/c:v', ns)
                        vals = [v.text or "-" for v in val_nodes]
                        series_data.append(vals)

                    # 4. Construct Markdown Table
                    md_table = f"\n\n### [Chart Data: {title}]\n"
                    md_table += "| Category | " + " | ".join(series_names) + " |\n"
                    md_table += "|---| " + " | ".join(["---"] * len(series_names)) + " |\n"

                    for i, cat in enumerate(categories):
                        row = [cat]
                        for s_data in series_data:
                            row.append(s_data[i] if i < len(s_data) else "-")
                        md_table += "| " + " | ".join(row) + " |\n"

                    chart_tables.append(md_table)

        except Exception as e:
            print(f"[Warning] Could not extract DOCX chart data: {e}")

        return chart_tables

    def parse(self, filepath: str) -> str:
        """Main execution flow: Convert -> Extract -> Analyze -> Format -> Clean."""
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File not found: {filepath}")

        ext = os.path.splitext(filepath)[1].lower()
        needs_conversion = ext in self.legacy_extensions
        target_file = filepath

        # 1. Convert legacy formats (.doc, .ppt, .odp) to .docx / .pptx
        if needs_conversion:
            target_file = self._convert_legacy_file(filepath, ext)

        try:
            # 2. Extract embedded images directly from the zip structure
            extracted_images = self.extract_office_images(target_file, self.media_temp_dir)

            # 3. Process images: Call Save_Image() and Vision_LLM() once per image
            descriptions_by_filename = {}
            for img_filename, local_path in extracted_images.items():
                if os.path.exists(local_path):
                    Save_Image(local_path)
                    descriptions_by_filename[img_filename] = Vision_LLM(local_path)

            # 4. Extract presentation and document charts as structured Markdown tables
            extracted_chart_tables = []
            if target_file.endswith(".pptx"):
                extracted_chart_tables = self._extract_pptx_charts_as_markdown(target_file)
            elif target_file.endswith(".docx"):
                extracted_chart_tables = self._extract_docx_charts_as_markdown(target_file)

            # 5. Parse main text layout and tables using Pandoc
            print(f"[System] Converting document to Markdown via Pandoc...")
            markdown_content = pypandoc.convert_file(
                target_file,
                "markdown",
                extra_args=[f"--extract-media={self.media_temp_dir}"],
            )

            # 6. Replace Image Tags in Markdown with Vision LLM Descriptions
            def replace_image_with_llm(match) -> str:
                img_src = match.group(2)
                img_filename = os.path.basename(urllib.parse.unquote(img_src))
                print(img_filename)
                if img_filename in descriptions_by_filename:
                    llm_desc = descriptions_by_filename[img_filename]
                    print(llm_desc)
                    return f"\n\n*[AI Image Description: {llm_desc}]*\n\n"

                return match.group(0)

            markdown_image_pattern = re.compile(r'!\[([^\]]*)\]\(([^)]+)\)(?:\{[^}]*\})?')
            final_content = re.sub(markdown_image_pattern, replace_image_with_llm, markdown_content)

            # 7. Replace OpenXML Chart Placeholders with Extracted Chart Markdown Tables
            chart_placeholder_pattern = re.compile(
                r"\\\[Graphic:\s*other:\s*http://schemas.openxmlformats.org/drawingml/2006/chart\\\]"
            )
            
            for chart_md in extracted_chart_tables:
                if chart_placeholder_pattern.search(final_content):
                    final_content = chart_placeholder_pattern.sub(chart_md, final_content, count=1)
                else:
                    # Append chart table at bottom if placeholder wasn't explicitly matched
                    final_content += f"\n{chart_md}"

            return final_content

        finally:
            # 8. Clean up temporary converted .docx/.pptx files
            if needs_conversion and os.path.exists(target_file):
                os.remove(target_file)
                print(f"[System] Cleaned up temporary converted file: {target_file}")

if __name__ == "__main__":
    parser = UniversalDocumentParser(media_temp_dir="document_media")
    test_file = "/home/user/Documents/Project_5/backend/experiments/data/code/documents/Shinchan_1.odt"

    if os.path.exists(test_file):
        try:
            parsed_markdown = parser.parse(test_file)
            print("\n--- FINAL PARSED CONTENT ---")
            print(parsed_markdown)
        except Exception as e:
            print(f"Error parsing: {e}")

[System] Converting legacy .odt to docx...
[System] Converted file saved uniquely as: /home/user/Documents/Project_5/backend/experiments/data/code/documents/Shinchan_1_1787839981210_5089.docx
[Log] Image saved permanently: document_media/word/media/image1.png
[Log] Calling Vision LLM for: document_media/word/media/image1.png
[Log] Image saved permanently: document_media/word/media/image4.png
[Log] Calling Vision LLM for: document_media/word/media/image4.png
[Log] Image saved permanently: document_media/word/media/image3.png
[Log] Calling Vision LLM for: document_media/word/media/image3.png
[Log] Image saved permanently: document_media/word/media/image2.png
[Log] Calling Vision LLM for: document_media/word/media/image2.png
[System] Converting document to Markdown via Pandoc...
image1.png
This is an AI-generated description of the image.
image2.png
This is an AI-generated description of the image.
image3.png
This is an AI-generated description of the image.
image4.png
This is an AI-gener

In [ ]:
import os
import re
import urllib.parse
import zipfile
import pypandoc
import xml.etree.ElementTree as ET
from pptx import Presentation

# Assuming these are defined elsewhere in your project
def Save_Image(local_path: str):
    pass

def Vision_LLM(local_path: str) -> str:
    print(local_path)
    return "Extracted Image Description"

class UniversalDocumentParser:
    def __init__(self, media_temp_dir: str = "temp_media"):
        self.media_temp_dir = media_temp_dir
        # pypandoc.download_pandoc()
        self.supported_extensions = {".docx", ".pptx", ".odt"}

    def extract_office_images(self, zip_filepath: str, output_dir: str) -> dict:
        """
        Extracts images directly from .docx, .pptx, and .odt zip structures.
        Returns a mapping of { image_filename: full_extracted_path }.
        """
        extracted_map = {}
        if not zipfile.is_zipfile(zip_filepath):
            return extracted_map

        with zipfile.ZipFile(zip_filepath, "r") as z:
            for member in z.namelist():
                # 'Pictures/' catches .odt images, 'word/media/' for docx, 'ppt/media/' for pptx
                if member.startswith(("word/media/", "ppt/media/", "Pictures/")) and not member.endswith("/"):
                    z.extract(member, path=output_dir)
                    full_path = os.path.join(output_dir, member)
                    filename = os.path.basename(member)
                    extracted_map[filename] = full_path

        return extracted_map

    def _extract_pptx_charts_as_markdown(self, pptx_path: str) -> list:
        """Reads PowerPoint charts via python-pptx."""
        chart_tables = []
        try:
            prs = Presentation(pptx_path)
            for slide_num, slide in enumerate(prs.slides, start=1):
                for shape in slide.shapes:
                    if shape.has_chart:
                        chart = shape.chart
                        title = chart.chart_title.text_frame.text if chart.has_title else f"Slide {slide_num} Chart"
                        categories = [str(c.label) for c in chart.plots[0].categories]
                        series_list = chart.series

                        series_names = [s.name if s.name else f"Series {i+1}" for i, s in enumerate(series_list)]
                        md_table = f"\n\n### [Chart Data: {title} (Slide {slide_num})]\n"
                        md_table += "| Category | " + " | ".join(series_names) + " |\n"
                        md_table += "|---| " + " | ".join(["---"] * len(series_names)) + " |\n"

                        for cat_idx, cat in enumerate(categories):
                            row_vals = []
                            for series in series_list:
                                try:
                                    row_vals.append(str(series.values[cat_idx]))
                                except IndexError:
                                    row_vals.append("-")
                            md_table += f"| {cat} | " + " | ".join(row_vals) + " |\n"

                        chart_tables.append(md_table)
        except Exception as e:
            print(f"[Warning] Could not extract PPTX chart data: {e}")

        return chart_tables

    def _extract_docx_charts_as_markdown(self, docx_path: str) -> list:
        """Reads Word charts natively from the DOCX XML zip structure."""
        chart_tables = []
        if not zipfile.is_zipfile(docx_path):
            return chart_tables

        try:
            with zipfile.ZipFile(docx_path, "r") as z:
                chart_files = [f for f in z.namelist() if f.startswith('word/charts/chart') and f.endswith('.xml')]
                for chart_num, chart_file in enumerate(chart_files, start=1):
                    root = ET.fromstring(z.read(chart_file))
                    ns = {'c': 'http://schemas.openxmlformats.org/drawingml/2006/chart'}

                    title_node = root.find('.//c:title//c:tx//c:v', ns)
                    title = title_node.text if title_node is not None else f"Chart {chart_num}"

                    series_nodes = root.findall('.//c:ser', ns)
                    if not series_nodes: continue

                    categories = [cat.text or "" for cat in series_nodes[0].findall('.//c:cat//c:pt/c:v', ns)]
                    series_names, series_data = [], []

                    for ser in series_nodes:
                        tx_node = ser.find('.//c:tx//c:v', ns)
                        series_names.append(tx_node.text if tx_node is not None else f"Series {len(series_names)+1}")
                        series_data.append([v.text or "-" for v in ser.findall('.//c:val//c:pt/c:v', ns)])

                    md_table = f"\n\n### [Chart Data: {title}]\n| Category | " + " | ".join(series_names) + " |\n"
                    md_table += "|---| " + " | ".join(["---"] * len(series_names)) + " |\n"

                    for i, cat in enumerate(categories):
                        row = [cat] + [s_data[i] if i < len(s_data) else "-" for s_data in series_data]
                        md_table += "| " + " | ".join(row) + " |\n"

                    chart_tables.append(md_table)
        except Exception as e:
            print(f"[Warning] Could not extract DOCX chart data: {e}")

        return chart_tables

    def _extract_odt_charts_as_markdown(self, odt_path: str) -> dict:
        """
        Reads charts directly from embedded objects in ODT files.
        Returns a mapping of { 'ObjectReplacements/Object X': markdown_table }
        """
        chart_map = {}
        if not zipfile.is_zipfile(odt_path):
            return chart_map

        try:
            with zipfile.ZipFile(odt_path, "r") as z:
                object_contents = [f for f in z.namelist() if f.startswith('Object ') and f.endswith('/content.xml')]
                
                for obj_file in object_contents:
                    root = ET.fromstring(z.read(obj_file))
                    ns = {'table': 'urn:oasis:names:tc:opendocument:xmlns:table:1.0',
                          'text': 'urn:oasis:names:tc:opendocument:xmlns:text:1.0'}
                    
                    tables = root.findall('.//table:table', ns)
                    if not tables: continue
                    
                    parsed_rows = []
                    for row in tables[0].findall('.//table:table-row', ns):
                        row_data = []
                        for cell in row.findall('.//table:table-cell', ns):
                            text_p = cell.find('.//text:p', ns)
                            # Safely extract text and ensure it is a string
                            val = text_p.text if text_p is not None else None
                            row_data.append(str(val).strip() if val else "-")
                        
                        if any(r != "-" for r in row_data):
                            parsed_rows.append(row_data)
                            
                    if len(parsed_rows) < 2: continue
                        
                    obj_name = obj_file.split('/')[0] # E.g., 'Object 1'
                    header = parsed_rows[0]
                    
                    md_table = f"\n\n### [Chart Data: {obj_name}]\n"
                    md_table += "| " + " | ".join(header) + " |\n"
                    md_table += "|---" * len(header) + "|\n"
                    
                    for row in parsed_rows[1:]:
                        # Pad row if it's missing trailing columns
                        padded_row = row + ["-"] * (len(header) - len(row))
                        md_table += "| " + " | ".join(padded_row) + " |\n"
                        
                    # Map to the exact path Pandoc uses in its placeholder
                    chart_map[f"ObjectReplacements/{obj_name}"] = md_table
                    
        except Exception as e:
            print(f"[Warning] Could not extract ODT chart data: {e}")
            
        return chart_map

    def parse(self, filepath: str) -> str:
        """Main execution flow: Extract -> Analyze -> Format -> Clean."""
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File not found: {filepath}")

        ext = os.path.splitext(filepath)[1].lower()
        if ext not in self.supported_extensions:
            raise ValueError(f"Unsupported extension: {ext}. Only .docx, .pptx, and .odt are supported.")

        # 1. Extract embedded images directly from the zip structure
        extracted_images = self.extract_office_images(filepath, self.media_temp_dir)

        # 2. Process images: Call Save_Image() and Vision_LLM() once per image
        descriptions_by_filename = {}
        for img_filename, local_path in extracted_images.items():
            if os.path.exists(local_path):
                Save_Image(local_path)
                descriptions_by_filename[img_filename] = Vision_LLM(local_path)

        # 3. Extract charts natively based on file type
        extracted_chart_tables_list = []
        extracted_odt_charts_map = {}
        
        if ext == ".pptx":
            extracted_chart_tables_list = self._extract_pptx_charts_as_markdown(filepath)
        elif ext == ".docx":
            extracted_chart_tables_list = self._extract_docx_charts_as_markdown(filepath)
        elif ext == ".odt":
            extracted_odt_charts_map = self._extract_odt_charts_as_markdown(filepath)

        # 4. Parse main text layout and tables using Pandoc
        print(f"[System] Converting {ext} to Markdown via Pandoc...")
        markdown_content = pypandoc.convert_file(
            filepath,
            "markdown",
            extra_args=[f"--extract-media={self.media_temp_dir}", "--quiet"]
        )

        # 5. Replace Standard Images with LLM Descriptions
        def replace_image_with_llm(match) -> str:
            img_src = match.group(2)
            img_filename = os.path.basename(urllib.parse.unquote(img_src))
            if img_filename in descriptions_by_filename:
                llm_desc = descriptions_by_filename[img_filename]
                return f"\n\n*[AI Image Description: {llm_desc}]*\n\n"
            return match.group(0)

        markdown_image_pattern = re.compile(r'!\[([^\]]*)\]\(([^)]+)\)(?:\{[^}]*\})?')
        final_content = re.sub(markdown_image_pattern, replace_image_with_llm, markdown_content)

        # 6. Replace OpenXML Chart Placeholders (.docx / .pptx)
        chart_placeholder_pattern = re.compile(r"\\\[Graphic:\s*other:\s*http://schemas.openxmlformats.org/drawingml/2006/chart\\\]")
        for chart_md in extracted_chart_tables_list:
            if chart_placeholder_pattern.search(final_content):
                final_content = chart_placeholder_pattern.sub(chart_md, final_content, count=1)
            else:
                final_content += f"\n{chart_md}"

        # 7. Replace Pandoc's ODT Object Placeholders (.odt)
        # Matches: []{.image .placeholderoriginal-image-src="./ObjectReplacements/Object 2" ... }
        def replace_odt_object(match):
            obj_src = match.group(1) # e.g., 'ObjectReplacements/Object 1'
            if obj_src in extracted_odt_charts_map:
                return extracted_odt_charts_map[obj_src]
            return "" # Clear the placeholder if we failed to parse it
            
        odt_placeholder_pattern = re.compile(r"\[\]\{[^}]*original-image-src=\"\.\/([^\"]+)\"[^}]*\}")
        final_content = re.sub(odt_placeholder_pattern, replace_odt_object, final_content)

        return final_content

if __name__ == "__main__":
    parser = UniversalDocumentParser(media_temp_dir="document_media")
    test_file = "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.pptx"

    if os.path.exists(test_file):
        try:
            parsed_markdown = parser.parse(test_file)
            print("\n--- FINAL PARSED CONTENT ---")
            print(parsed_markdown)
        except Exception as e:
            print(f"Error parsing: {e}")

document_media/ppt/media/image1.png
[System] Converting .pptx to Markdown via Pandoc...

--- FINAL PARSED CONTENT ---
## This is the slide a PPTX slide {#slide-1}

Don't dare and fuck you if you try to delete it it should state how it is 

## This is the image content of this file {#slide-2}



*[AI Image Description: Extracted Image Description]*



## This is the table {#slide-3}

  1   Delhi         New Delhi   India   110001
  --- ------------- ----------- ------- --------
  2   Maharashtra   Mumbai      India   400001
  3   Tamil Nadu    Chennai     India   600001
  4   Arunachal     Itanagar    India   791111

## This is Chart {#slide-4}



### [Chart Data: Popullation Chart
A chart showing popullation for 3 years  (Slide 4)]
| Category | Column 1 | Column 2 | Column 3 |
|---| --- | --- | --- |
| Row 1 | 9.1 | 3.2 | 4.54 |
| Row 2 | 2.4 | 8.8 | 9.65 |
| Row 3 | 3.1 | 1.5 | 3.7 |
| Row 4 | 4.3 | 9.02 | 6.2 |


## This is the fifth slide {#slide-5}

- Content floating around



*[A